In [1]:
import sqlite3
import pandas as pd
import json
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from dataclasses import dataclass
from typing import Dict, Tuple

sns.set_style('whitegrid')
%matplotlib inline

In [2]:
# Connect to the database
conn = sqlite3.connect('../data/teaser_scores/anomaly_scores.db')

df = pd.read_sql_query('SELECT * FROM anomaly_scores', conn)

print(f"Total commits: {len(df)}")
df.head()

Total commits: 2760


,commit_node,origin_url,commit_swhid,large_addition,large_deletion,large_modification,large_moves,time_of_day,author_tod,author_tod_global,...,newcomer_committer,account_age_author,account_age_committer,repository_dormancy,sensitive_file,prebuilt_file,ownership_score_json,ownership_has_unknown,total_score,weighted_score
0,4587448,https://github.com/Avik-Jain/100-Days-Of-ML-Code,swh:1:rev:dfbc332b27bd3347c94e0ca1ea43ddb8f380...,0.0,0.0,0.284722,0.0,0.395992,0.323473,0.418392,...,0.0,0.0,0.0,1.0,0.0,0.0,"{""README.md"":0.8823529411764706}",1,0.211757,0.211757
1,4587652,https://github.com/Avik-Jain/100-Days-Of-ML-Code,swh:1:rev:19d2c0342bb53dab77016bfcadac90f70ab1...,0.0,0.0,0.284722,0.0,0.403336,0.306164,0.306164,...,0.0,0.0,0.0,1.0,0.0,0.0,{},0,0.178758,0.178758
2,4587653,https://github.com/Avik-Jain/100-Days-Of-ML-Code,swh:1:rev:62d67a97bccc8390e69531373ef942048a68...,0.0,0.0,0.284722,0.0,0.403336,0.306164,0.306164,...,0.0,0.0,0.0,1.0,0.0,0.0,{},0,0.178758,0.178758
3,4587654,https://github.com/Avik-Jain/100-Days-Of-ML-Code,swh:1:rev:8ff74a7efd391ed5670f0fc239ff96794087...,0.0,0.0,0.284722,0.0,0.403336,0.306164,0.306164,...,0.0,0.0,0.0,1.0,0.0,0.0,{},0,0.178758,0.178758
4,4587677,https://github.com/Avik-Jain/100-Days-Of-ML-Code,swh:1:rev:fcc10ef5162522b5888d80904fe66f8eef09...,1.0,0.0,0.000000,0.0,0.403336,0.306164,0.306164,...,0.0,0.0,0.0,1.0,0.0,0.0,{},0,0.294090,0.294090


In [3]:
@dataclass
class ScoreThresholds:
    max_normal_score: float  # Scores <= this are considered normal
    min_suspicious_score: float  # Scores >= this are considered suspicious
    
    def classify(self, score: float) -> str:
        if score <= self.max_normal_score:
            return "normal"
        elif score >= self.min_suspicious_score:
            return "suspicious"
        else:
            return "undetermined"

thresholds = ScoreThresholds(max_normal_score=0.3, min_suspicious_score=0.7)
print(f"Thresholds: normal <= {thresholds.max_normal_score}, suspicious >= {thresholds.min_suspicious_score}")

Thresholds: normal <= 0.3, suspicious >= 0.7


In [ ]:
@dataclass
class AnomalyWeights:
    large_addition: float = 1.0
    large_deletion: float = 1.0
    large_modification: float = 1.0
    large_moves: float = 1.0
    time_of_day: float = 1.0
    author_tod: float = 1.0
    author_tod_global: float = 1.0
    committer_tod: float = 1.0
    committer_tod_global: float = 1.0
    file_type_interaction: float = 1.0
    file_type_interaction_author: float = 1.0
    number_commits_author: float = 1.0
    number_commits_committer: float = 1.0
    newcomer_author: float = 1.0
    newcomer_committer: float = 1.0
    account_age_author: float = 1.0
    account_age_committer: float = 1.0
    repository_dormancy: float = 1.0
    sensitive_file: float = 1.0
    prebuilt_file: float = 1.0
    
    WEIGHT_MIN: float = -2.0
    WEIGHT_MAX: float = 2.0
    
    def get_weight_dict(self) -> Dict[str, float]:
        return {
            'large_addition': self.large_addition,
            'large_deletion': self.large_deletion,
            'large_modification': self.large_modification,
            'large_moves': self.large_moves,
            'time_of_day': self.time_of_day,
            'author_tod': self.author_tod,
            'author_tod_global': self.author_tod_global,
            'committer_tod': self.committer_tod,
            'committer_tod_global': self.committer_tod_global,
            'file_type_interaction': self.file_type_interaction,
            'file_type_interaction_author': self.file_type_interaction_author,
            'number_commits_author': self.number_commits_author,
            'number_commits_committer': self.number_commits_committer,
            'newcomer_author': self.newcomer_author,
            'newcomer_committer': self.newcomer_committer,
            'account_age_author': self.account_age_author,
            'account_age_committer': self.account_age_committer,
            'repository_dormancy': self.repository_dormancy,
            'sensitive_file': self.sensitive_file,
            'prebuilt_file': self.prebuilt_file,
        }
    
    def compute_weighted_score(self, row: pd.Series) -> float:
        weights_dict = self.get_weight_dict()
    
        total = 0.0
        weight_sum = 0.0
        
        for feature, weight in weights_dict.items():
            if feature in row and pd.notna(row[feature]):
                total += row[feature] * weight
                weight_sum += abs(weight)
                
        if weight_sum > 0.0:
            return total / weight_sum
        return 0.0
    
    def clamp(self):
        for field in self.__dataclass_fields__:
            if field not in ['WEIGHT_MIN', 'WEIGHT_MAX']:
                current = getattr(self, field)
                setattr(self, field, np.clip(current, self.WEIGHT_MIN, self.WEIGHT_MAX))
    
    def randomize(self, min_val: float = None, max_val: float = None):
        if min_val is None:
            min_val = self.WEIGHT_MIN
        if max_val is None:
            max_val = self.WEIGHT_MAX
            
        for field in self.__dataclass_fields__:
            if field not in ['WEIGHT_MIN', 'WEIGHT_MAX']:
                setattr(self, field, np.random.uniform(min_val, max_val))
    
    def perturb(self, temperature: float, min_val: float = None, max_val: float = None):
        if min_val is None:
            min_val = self.WEIGHT_MIN
        if max_val is None:
            max_val = self.WEIGHT_MAX
            
        for field in self.__dataclass_fields__:
            if field not in ['WEIGHT_MIN', 'WEIGHT_MAX']:
                current = getattr(self, field)
                # Perturbation proportional to temperature
                delta = np.random.normal(-(temperature * 0.5), temperature * 0.5)
                new_val = np.clip(current + delta, min_val, max_val)
                setattr(self, field, new_val)
    
    def copy(self):
        return AnomalyWeights(**self.get_weight_dict())

# Initialize default weights
weights = AnomalyWeights()
print(f"Initialized weights (all set to 1.0)")
print(f"Weight range: [{weights.WEIGHT_MIN}, {weights.WEIGHT_MAX}]")

## Labeled Data

In [4]:
# Example: Create sample labeled data
# In practice, you would manually label commits or use known anomalies

# For demonstration, let's assume:
# - High total_score commits are suspicious
# - Low total_score commits are normal

# Get score columns (exclude metadata columns)
score_columns = [col for col in df.columns if col not in 
                 ['commit_node', 'origin_url', 'commit_swhid', 'ownership_score_json', 
                  'ownership_has_unknown', 'total_score', 'weighted_score']]

print(f"Score columns: {score_columns}")
print(f"\nTotal commits in dataset: {len(df)}")

# Example labeling: use percentiles
# Bottom 20% = normal, Top 20% = suspicious
normal_threshold = df['total_score'].quantile(0.20)
suspicious_threshold = df['total_score'].quantile(0.80)

df['label'] = 'undetermined'
df.loc[df['total_score'] <= normal_threshold, 'label'] = 'normal'
df.loc[df['total_score'] >= suspicious_threshold, 'label'] = 'suspicious'

print(f"\nLabel distribution:")
print(df['label'].value_counts())
print(f"\nNormal threshold: {normal_threshold:.3f}")
print(f"Suspicious threshold: {suspicious_threshold:.3f}")

Score columns: ['large_addition', 'large_deletion', 'large_modification', 'large_moves', 'time_of_day', 'author_tod', 'author_tod_global', 'committer_tod', 'committer_tod_global', 'file_type_interaction', 'file_type_interaction_author', 'number_commits_author', 'number_commits_committer', 'newcomer_author', 'newcomer_committer', 'account_age_author', 'account_age_committer', 'repository_dormancy', 'sensitive_file', 'prebuilt_file']

Total commits in dataset: 2760

Label distribution:
label
undetermined    1653
suspicious       555
normal           552
Name: count, dtype: int64

Normal threshold: 0.183
Suspicious threshold: 0.316


## Simulated Annealing Algorithm

The objective is to maximize the distance between:
- `max(normal_scores)`: highest score among normal commits
- `min(suspicious_scores)`: lowest score among suspicious commits

In [ ]:
def compute_separation_score(weights: AnomalyWeights, 
                             df_normal: pd.DataFrame, 
                             df_suspicious: pd.DataFrame,
                             score_columns: list) -> Tuple[float, float, float]:
    # Compute weighted scores for all commits
    normal_scores = df_normal.apply(lambda row: weights.compute_weighted_score(row), axis=1)
    suspicious_scores = df_suspicious.apply(lambda row: weights.compute_weighted_score(row), axis=1)
    
    if len(normal_scores) == 0 or len(suspicious_scores) == 0:
        return -float('inf'), 0.0, 1.0
    
    max_normal = normal_scores.max()
    min_suspicious = suspicious_scores.min()
    separation = min_suspicious - max_normal
    
    return separation, max_normal, min_suspicious


def simulated_annealing(df: pd.DataFrame,
                       score_columns: list,
                       initial_temp: float = 1.0,
                       cooling_rate: float = 0.95,
                       min_temp: float = 0.001,
                       max_iterations: int = 1000) -> Tuple[AnomalyWeights, list]:
    """
    Use simulated annealing to find optimal weights that maximize separation
    between normal and suspicious commits.
    
    Args:
        df: DataFrame with labeled commits
        score_columns: List of score column names
        initial_temp: Starting temperature
        cooling_rate: Temperature reduction factor (0 < rate < 1)
        min_temp: Minimum temperature (stopping criterion)
        max_iterations: Maximum number of iterations
    
    Returns:
        - best_weights: Optimal weights found
        - history: List of (iteration, temperature, separation, accepted) tuples
    """
    # Split data by label
    df_normal = df[df['label'] == 'normal'][score_columns]
    df_suspicious = df[df['label'] == 'suspicious'][score_columns]
    
    print(f"Normal commits: {len(df_normal)}")
    print(f"Suspicious commits: {len(df_suspicious)}")
    
    # Initialize with random weights
    current_weights = AnomalyWeights()
    current_weights.randomize()
    
    current_separation, current_max_normal, current_min_suspicious = \
        compute_separation_score(current_weights, df_normal, df_suspicious, score_columns)
    
    best_weights = current_weights.copy()
    best_separation = current_separation
    
    # Track history
    history = []
    temperature = initial_temp
    iteration = 0
    
    print(f"\nStarting simulated annealing...")
    print(f"Weight range: [{current_weights.WEIGHT_MIN}, {current_weights.WEIGHT_MAX}]")
    print(f"Initial separation: {current_separation:.4f}")
    print(f"  max(normal) = {current_max_normal:.4f}, min(suspicious) = {current_min_suspicious:.4f}")
    
    while temperature > min_temp and iteration < max_iterations:
        iteration += 1
        
        # Create candidate
        candidate_weights = current_weights.copy()
        candidate_weights.perturb(temperature)
        
        # Evaluate candidate
        candidate_separation, cand_max_normal, cand_min_suspicious = \
            compute_separation_score(candidate_weights, df_normal, df_suspicious, score_columns)
        
        delta = candidate_separation - current_separation
        
        if delta > 0:
            # Better solution, always accept
            accept_prob = 1.0
        else:
            # Worse solution, accept with probability based on temperature
            accept_prob = np.exp(delta / temperature)
        
        accepted = np.random.random() < accept_prob
        
        if accepted:
            current_weights = candidate_weights
            current_separation = candidate_separation
            current_max_normal = cand_max_normal
            current_min_suspicious = cand_min_suspicious
            if current_separation > best_separation:
                best_weights = current_weights.copy()
                best_separation = current_separation
        
        history.append((iteration, temperature, current_separation, accepted))
        
        temperature *= cooling_rate
        
        # Print progress every 100 iterations
        if iteration % 100 == 0:
            print(f"Iter {iteration}: T={temperature:.4f}, "
                  f"sep={current_separation:.4f}, "
                  f"best_sep={best_separation:.4f}, "
                  f"accept_rate={sum(h[3] for h in history[-100:]) / 100:.2f}")
    
    print(f"\nOptimization complete!")
    print(f"Best separation: {best_separation:.4f}")
    
    best_weights.clamp()
    
    final_sep, final_max_normal, final_min_suspicious = \
        compute_separation_score(best_weights, df_normal, df_suspicious, score_columns)
    
    print(f"  max(normal) = {final_max_normal:.4f}")
    print(f"  min(suspicious) = {final_min_suspicious:.4f}")
    print(f"  gap = {final_sep:.4f}")
    
    return best_weights, history

## Run the Algorithm

In [ ]:
optimal_weights, history = simulated_annealing(
    df=df,
    score_columns=score_columns,
    initial_temp=1.0,
    cooling_rate=0.95,
    min_temp=0.001,
    max_iterations=1000
)

print("\n" + "="*60)
print("OPTIMAL WEIGHTS:")
print("="*60)
for feature, weight in optimal_weights.get_weight_dict().items():
    print(f"{feature:30s}: {weight:.4f}")

## Visualize Optimization Progress

In [ ]:
# Convert history to DataFrame for easier plotting
history_df = pd.DataFrame(history, columns=['iteration', 'temperature', 'separation', 'accepted'])

# Create figure with subplots
fig, axes = plt.subplots(3, 1, figsize=(12, 10))

# Plot 1: Separation over iterations
axes[0].plot(history_df['iteration'], history_df['separation'], alpha=0.7)
axes[0].set_xlabel('Iteration')
axes[0].set_ylabel('Separation Score')
axes[0].set_title('Separation Between Normal and Suspicious Commits')
axes[0].grid(True, alpha=0.3)

# Plot 2: Temperature decay
axes[1].plot(history_df['iteration'], history_df['temperature'], color='orange')
axes[1].set_xlabel('Iteration')
axes[1].set_ylabel('Temperature')
axes[1].set_title('Temperature Decay (Simulated Annealing)')
axes[1].grid(True, alpha=0.3)

# Plot 3: Acceptance rate (rolling window)
window_size = 50
history_df['acceptance_rate'] = history_df['accepted'].rolling(window=window_size, min_periods=1).mean()
axes[2].plot(history_df['iteration'], history_df['acceptance_rate'], color='green')
axes[2].set_xlabel('Iteration')
axes[2].set_ylabel('Acceptance Rate')
axes[2].set_title(f'Acceptance Rate (Rolling Window = {window_size})')
axes[2].grid(True, alpha=0.3)
axes[2].set_ylim([0, 1])

plt.tight_layout()
plt.show()

## Visualize Weight Distribution

In [ ]:
# Plot optimal weights as a bar chart
weights_dict = optimal_weights.get_weight_dict()
features = list(weights_dict.keys())
weight_values = list(weights_dict.values())

plt.figure(figsize=(12, 6))
colors = ['red' if w > 1.5 else 'orange' if w > 1.0 else 'green' for w in weight_values]
plt.bar(range(len(features)), weight_values, color=colors, alpha=0.7)
plt.xticks(range(len(features)), features, rotation=45, ha='right')
plt.ylabel('Weight Value')
plt.title('Optimal Weights for Each Anomaly Score Component')
plt.axhline(y=1.0, color='blue', linestyle='--', alpha=0.5, label='Baseline (1.0)')
plt.legend()
plt.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

# Show top 5 most important features
sorted_weights = sorted(weights_dict.items(), key=lambda x: x[1], reverse=True)
print("\nTop 5 Most Important Features:")
for i, (feature, weight) in enumerate(sorted_weights[:5], 1):
    print(f"{i}. {feature:30s}: {weight:.4f}")

## Compare Score Distributions: Before and After Optimization

In [ ]:
# Compute optimized scores for all commits
df['optimized_score'] = df[score_columns].apply(
    lambda row: optimal_weights.compute_weighted_score(row), axis=1
)

# Create comparison plots
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Plot 1: Original scores
for label in ['normal', 'suspicious']:
    subset = df[df['label'] == label]['total_score']
    axes[0].hist(subset, alpha=0.6, bins=30, label=label)
axes[0].set_xlabel('Original Total Score')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Original Score Distribution')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Plot 2: Optimized scores
for label in ['normal', 'suspicious']:
    subset = df[df['label'] == label]['optimized_score']
    axes[1].hist(subset, alpha=0.6, bins=30, label=label)
axes[1].set_xlabel('Optimized Weighted Score')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Optimized Score Distribution')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Print statistics
print("Original Scores:")
print(f"  Normal:     mean={df[df['label']=='normal']['total_score'].mean():.4f}, "
      f"max={df[df['label']=='normal']['total_score'].max():.4f}")
print(f"  Suspicious: mean={df[df['label']=='suspicious']['total_score'].mean():.4f}, "
      f"min={df[df['label']=='suspicious']['total_score'].min():.4f}")
print(f"  Separation: {df[df['label']=='suspicious']['total_score'].min() - df[df['label']=='normal']['total_score'].max():.4f}")

print("\nOptimized Scores:")
print(f"  Normal:     mean={df[df['label']=='normal']['optimized_score'].mean():.4f}, "
      f"max={df[df['label']=='normal']['optimized_score'].max():.4f}")
print(f"  Suspicious: mean={df[df['label']=='suspicious']['optimized_score'].mean():.4f}, "
      f"min={df[df['label']=='suspicious']['optimized_score'].min():.4f}")
print(f"  Separation: {df[df['label']=='suspicious']['optimized_score'].min() - df[df['label']=='normal']['optimized_score'].max():.4f}")